In [ ]:
# Feature extraction con CSP e epoch e classificazione binaria (riposo vs attivazione). 
# Classificatore SVM. Le performance sono chiaramente migliori rispetto alle windows, ma ha poca importanza.
# Ultimo update: csp viene eseguito singolarmente per ogni paziente e per ognugno è addestrato un classificatore SVM, questo perchè
# csp se la cava molto male tra pazienti diversi.

import mne
from mne.decoding import CSP
from mne_bids import BIDSPath, read_raw_bids
import numpy as np
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from pathlib import Path
import matplotlib.pyplot as plt


mne.set_log_level('WARNING')

root = "../data"
root = (Path(root).resolve())
runs = ["4", "8", "12"]   # Le run che vengono prese in considerazione
train_runs = ["4", "8"]
test_run = "12" 

# Eseguiamo la scansione di tutti i soggetti 
for i in range(1, 10):
    subject = f"{i:03d}"
    
    X_train = []
    y_train = []

    X_test = []
    y_test = []
    # Per ogni soggetto eseguiamo la scansione sulle run di nostro interesse
    for run in runs:
        bids_path = BIDSPath(   # Specifichiamo il percorso del dataset e BIDS eseguirà correttamente la scansione 
            subject=subject,
            task="motion",
            run=run,
            datatype="eeg",
            root=root,
        )
        
        try:
            # Fase 1: lettura dei dati e pre-processing
            raw = read_raw_bids(bids_path, verbose=False)  
            events, event_id = mne.events_from_annotations(raw, verbose=False) 
            raw.load_data(verbose=False) # Carico i dati in memoria per poter filtrare ecc.
            raw.filter(l_freq=1, h_freq=30, verbose=False)  # Filtro passa banda 1-30 Hz
            #raw.set_eeg_reference('average', projection=False, verbose=False)  # Riferimento medio
        
            epochs = mne.Epochs(
                raw,
                events,
                event_id=event_id,
                tmin=0.0,
                tmax=4.0,
                baseline=(0.0, 0.0),
                preload=True,
                verbose=False
            )          

            # Divido la classificazione in due step: per ora distinguo tra stato di riposo e di attivazione
            y = epochs.events[:, 2]

            # Etichette rest/active
            y_rest_active = (y != 1).astype(int)

            # Etichette destra/sinistra
            mask = y != 1
            epochs_lr = epochs[mask]
            y_lr = np.where(epochs_lr.events[:, 2] == 2, 0, 1)

            # Assegna a y_binary il valore desiderato
            y_binary = y_lr

            if run in train_runs:
                X_train.append(epochs.get_data())
                y_train.append(y_binary)
            else:
                X_test.append(epochs.get_data())
                y_test.append(y_binary)
        
            # print(f"Soggetto {subject} run {run} - Campioni: {X_csp.shape[0]}, Feature per campione: {X_csp.shape[1]}, Etichette: {y.shape[0]}")

        except Exception as e:
            print(f"Errore {subject}: {e}")

    # Per ogni paziente addestriamo un modello con CSP e SVM
    X_train = np.concatenate(X_train, axis=0)
    y_train = np.concatenate(y_train)

    X_test = np.concatenate(X_test, axis=0)
    y_test = np.concatenate(y_test)

    pipe = Pipeline([
        ("csp", CSP()),
        ("svm", SVC())
    ])

    param_grid = {
        "csp__n_components": [2, 4, 6, 8],
        "csp__log": [True],
        "svm__C": [0.1, 1, 10, 100],
        "svm__kernel": ["linear", "rbf"]
    }

    grid = GridSearchCV(
        pipe,
        param_grid,
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)
    print(f"Best score: {grid.best_score_:4f}")
    print("Best params:", grid.best_params_)

    test_accuracy = grid.score(X_test, y_test)
    print(f"Accuratezza reale su RUN 12 (Test Set): {test_accuracy:.4f}")


KeyboardInterrupt: 